# loehoer.ai — AI Pipeline untuk Generate PRD

Notebook ini mendemonstrasikan alur AI dari awal hingga akhir: data mentah, chunking, embedding, retrieval, hingga LLM menghasilkan PRD.

**Pipeline:**
1. Data Ingestion — download dari Google Drive, konversi PDF/DOCX/PPTX ke teks, chunking 800 karakter, embedding, simpan ke ChromaDB
2. Retrieval — semantic search untuk mencari dokumen relevan berdasarkan query
3. Generation — Llama 3.2 1B Instruct menghasilkan PRD berdasarkan referensi + template

Kode yang digunakan identik dengan modul `App/` (backend FastAPI).

> **Peran dalam UAS Kecerdasan Buatan:** **Model Utama** — pendekatan *Retrieval-Augmented Generation* (RAG). Mewakili solusi utama yang dibandingkan dengan Model Pembanding (Tanpa RAG, `UAS_Model/Comparison_model.ipynb`). Evaluasi ROUGE pada `Laporan_uas.md`.


---
## 1. Persiapan (Setup)

Import semua modul yang diperlukan dan setup path project.

In [ ]:
import sys, os, time, shutil
from pathlib import Path

# Cari root project secara otomatis
def _find_project_root():
    cur = Path.cwd().resolve()
    for _ in range(10):
        if (cur / 'App' / 'chatbot.py').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return Path.cwd()

BASE_DIR = _find_project_root()
sys.path.insert(0, str(BASE_DIR))
sys.path.insert(0, str(BASE_DIR / 'App'))

import config
from rag_builder import get_vectorstore
from chatbot import PRDChatbot, _get_device, TEMPLATES

print(f'Project root : {BASE_DIR}')
print(f'Model LLM    : {config.LLM_MODEL_NAME}')
print(f'Embedding    : {config.EMBEDDING_MODEL}')
print(f'Device       : {_get_device()}')


---
## 2. Data Ingestion — Vectorstore

Membangun basis pengetahuan dari dokumen referensi:
- Download file dari Google Drive
- Konversi PDF, DOCX, PPTX ke teks
- Chunking 800 karakter per segmen
- Embedding dengan all-MiniLM-L6-v2
- Simpan di ChromaDB

Fungsi `get_vectorstore()` otomatis memuat yang sudah ada, atau membangun baru.

In [ ]:
vs = get_vectorstore()

if vs:
    count = vs._collection.count()
    print(f'ChromaDB siap: {count} chunks')
    print(f'Lokasi: {config.VECTORSTORE_DIR}')
    ref_count = len(list((config.BASE_DIR / "data" / "dataset").glob("*.md")))
    print(f'Sumber: {ref_count} file PRD referensi dari data/dataset/')
else:
    print('Gagal membangun vectorstore.')


In [ ]:
raw_dir = BASE_DIR / 'data' / 'dataset'
ref_files = sorted(raw_dir.glob('*.md')) if raw_dir.exists() else []

print(f'Jumlah file referensi: {len(ref_files)}')
for f in ref_files:
    name = f.stem.replace('_', ' ').title()
    size = len(f.read_text())
    print(f'  {name} ({size:,} chars)')


---
## 3. Retrieval — Semantic Search

Mencari dokumen relevan dari ChromaDB menggunakan semantic search (berdasarkan makna, bukan keyword).

Parameter `RAG_TOP_K_EKSEKUSI` menentukan jumlah dokumen yang diambil.

In [ ]:
query = "aplikasi mobile laundry antar jemput"
k = config.RAG_TOP_K_EKSEKUSI
docs = vs.similarity_search(query, k=k)

print(f'Query: "{query}"')
print(f'Dokumen relevan (top {k}):\n')
for i, d in enumerate(docs):
    src = d.metadata.get('source', '?')
    for ext in ['.md', '.pdf', '.pptx', '.docx']:
        src = src.replace(ext, '')
    src = src.replace('_', ' ').title()
    print(f'  [{i+1}] {src} (chunk {d.metadata["chunk_id"]})')
    print(f'      -> {d.page_content[:200].strip()}...')
    print()


---
## 4. Generation — Load Model & Generate PRD

Memuat Llama 3.2 1B Instruct. Proses ini memakan waktu 10-30 detik (lebih cepat jika sudah tersimpan di `Model/llama/`).

Setelah model siap, kita bisa:
- **generate_prd()** — tunggu hasil akhir
- **generate_streaming()** — lihat output token per token

### Intent Detection
AI otomatis mendeteksi maksud user:
- "Apa itu PRD?" -> intent = diskusi -> AI menjelaskan
- "Buat PRD untuk laundry" -> intent = generate -> AI membuat PRD

### General Knowledge
Jika referensi RAG minim (< 200 chars), AI otomatis menggunakan pengetahuan umum untuk melengkapi jawaban.

In [ ]:
print(f'Loading model ({config.LLM_MODEL_NAME})...')
print(f'Device: {_get_device()}')
if config.LOCAL_MODEL_DIR.exists():
    print(f'Model lokal: {config.LOCAL_MODEL_DIR}')
else:
    print(f'Download dari HuggingFace (sekali, ~2.5GB)...')

_chatbot = PRDChatbot()
print(f'Model siap. Device: {_chatbot.device}')


---
## 5. Demo — Pipeline Lengkap

### 5.1 Fungsi Helper

Dua cara memanggil AI:
- **generate_prd** — tunggu hasil akhir
- **generate_streaming** — lihat output token per token (real-time)

In [ ]:
def _get_chatbot():
    g = globals()
    if '_chatbot' not in g:
        print('Loading model...')
        g['_chatbot'] = PRDChatbot()
    return g['_chatbot']

def generate_prd(user_input, template_key='master'):
    return _get_chatbot().generate_prd(user_input, template_key=template_key)

def generate_streaming(user_input, template_key='master'):
    cb = _get_chatbot()
    cb.generate_prd_async(user_input, template_key=template_key)
    full = ''
    while not cb.is_done():
        if cb.get_error():
            break
        chunk = cb.get_partial()[len(full):]
        if chunk:
            full += chunk
        time.sleep(0.05)
    err = cb.get_error()
    if err:
        yield f'[Error: {err}]'

print('Fungsi siap.')
print('  generate_prd("...", template_key="master")')
print('  generate_streaming("...", template_key="startup")')


### 5.2 Template PRD

Terdapat 5 template yang bisa digunakan:

In [ ]:
print(f'Tersedia {len(TEMPLATES)} template:\n')
for key, t in TEMPLATES.items():
    print(f'  [{key:<12}] {t["label"]}')
    print(f'               {t["desc"]}')
    print()


### 5.3 Demo Intent Detection (Tanya Jawab)

AI membedakan pertanyaan ("Apa itu PRD?") dengan perintah generate ("Buat PRD...").

In [ ]:
prompt = "Apa itu PRD dan kenapa penting untuk pengembangan produk?"
print(f'User: "{prompt}"')
print(f'Intent: diskusi -> AI menjelaskan\n')

t0 = time.time()
hasil = generate_prd(prompt)  # auto-detect intent -> diskusi
print(hasil)
print(f'\n--- {time.time()-t0:.1f}s | {len(hasil)} chars ---')


### 5.4 Demo Generate PRD — 5 Template

Generate PRD menggunakan 5 template berbeda.

In [ ]:
# Demo 1: Template Master
prompt = "Buat PRD untuk aplikasi mobile laundry antar jemput"
print(f'Prompt: "{prompt}"')
print(f'Template: {TEMPLATES["master"]["label"]}\n')

t0 = time.time()
hasil = generate_prd(prompt, template_key='master')
print(hasil)
print(f'\n--- {time.time()-t0:.1f}s | {len(hasil)} chars ---')


In [ ]:
# Demo 2: Template Startup
prompt = "Buat PRD untuk platform belajar coding online interaktif"
print(f'Prompt: "{prompt}"')
print(f'Template: {TEMPLATES["startup"]["label"]}\n')

t0 = time.time()
hasil = generate_prd(prompt, template_key='startup')
print(hasil)
print(f'\n--- {time.time()-t0:.1f}s | {len(hasil)} chars ---')


In [ ]:
# Demo 3: Template Mobile
prompt = "Buat PRD untuk fitur push notification di aplikasi e-commerce"
print(f'Prompt: "{prompt}"')
print(f'Template: {TEMPLATES["mobile"]["label"]}\n')

t0 = time.time()
hasil = generate_prd(prompt, template_key='mobile')
print(hasil)
print(f'\n--- {time.time()-t0:.1f}s | {len(hasil)} chars ---')


In [ ]:
# Demo 4: Template Enterprise
prompt = "Buat PRD untuk sistem manajemen inventaris gudang berbasis IoT"
print(f'Prompt: "{prompt}"')
print(f'Template: {TEMPLATES["enterprise"]["label"]}\n')

t0 = time.time()
hasil = generate_prd(prompt, template_key='enterprise')
print(hasil)
print(f'\n--- {time.time()-t0:.1f}s | {len(hasil)} chars ---')


In [ ]:
# Demo 5: Template Data
prompt = "Buat PRD untuk dashboard monitoring penjualan real-time"
print(f'Prompt: "{prompt}"')
print(f'Template: {TEMPLATES["data"]["label"]}\n')

t0 = time.time()
hasil = generate_prd(prompt, template_key='data')
print(hasil)
print(f'\n--- {time.time()-t0:.1f}s | {len(hasil)} chars ---')


### 5.5 Demo Streaming

Output token per token (real-time), sama seperti di frontend React.

In [ ]:
prompt = "Buat PRD untuk aplikasi manajemen tugas tim"
print(f'Streaming: "{prompt}"')
print(f'Template: {TEMPLATES["master"]["label"]}\n')

t0 = time.time()
full = ''
for chunk in generate_streaming(prompt, template_key='master'):
    print(chunk, end='', flush=True)
    full += chunk
print(f'\n\n--- {time.time()-t0:.1f}s | {len(full)} chars ---')


### 5.6 Simpan Output ke File

In [ ]:
output_dir = BASE_DIR / 'output'
output_dir.mkdir(exist_ok=True)

hasil = generate_prd("Buat PRD untuk platform e-commerce frozen food B2B", template_key='startup')

path = output_dir / 'prd_frozen_food_b2b.md'
path.write_text(hasil)
print(f'PRD disimpan ke: {path}')
print(f'Ukuran: {len(hasil):,} chars')
print(f'\nPreview:\n{"-"*50}\n{hasil[:300]}')


---
## 6. Evaluasi ROUGE

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) mengukur kemiripan PRD hasil AI dengan dokumen referensi.

**Metrik:**
- ROUGE-1: overlap unigram (kata individu)
- ROUGE-2: overlap bigram (pasangan kata)
- ROUGE-L: longest common subsequence (struktur kalimat)

Precision: proporsi output AI yang ada di referensi.
Recall: proporsi referensi yang ditulis ulang oleh AI.
F1: harmonic mean precision & recall.

### Hasil Pre-computed

Hasil ROUGE untuk **Contoh PRD Aplikasi Keuangan** (Reference: 1,205 chars | Output AI: 2,742 chars):

| Metrik   | Precision | Recall | F1     |
|----------|-----------|--------|--------|
| ROUGE-1  | 0.2919    | 0.6235 | 0.3976 |
| ROUGE-2  | 0.2203    | 0.4720 | 0.3004 |
| ROUGE-L  | 0.2717    | 0.5802 | 0.3701 |

Hasil ROUGE untuk **Contoh PRD Ecommerce** (Reference: 1,288 chars | Output AI: 2,520 chars):

| Metrik   | Precision | Recall | F1     |
|----------|-----------|--------|--------|
| ROUGE-1  | 0.3505    | 0.6000 | 0.4425 |
| ROUGE-2  | 0.2586    | 0.4438 | 0.3268 |
| ROUGE-L  | 0.3058    | 0.5235 | 0.3861 |

Jalankan kode di bawah untuk menghitung ulang dengan hasil yang berbeda (generasi bersifat non-deterministik).

In [ ]:
# Install rouge-score jika belum ada
try:
    import importlib
    importlib.import_module('rouge_score')
except ImportError:
    import subprocess
    print('Install rouge-score...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rouge-score', '-q'])

from rouge_score import rouge_scorer


def evaluate_rouge(hypothesis: str, reference: str) -> dict:
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, hypothesis)
    return {
        'ROUGE-1': {'P': scores['rouge1'].precision, 'R': scores['rouge1'].recall, 'F1': scores['rouge1'].fmeasure},
        'ROUGE-2': {'P': scores['rouge2'].precision, 'R': scores['rouge2'].recall, 'F1': scores['rouge2'].fmeasure},
        'ROUGE-L': {'P': scores['rougeL'].precision, 'R': scores['rougeL'].recall, 'F1': scores['rougeL'].fmeasure},
    }


def print_rouge_table(scores: dict, label: str = ''):
    if label:
        print(f'\n=== {label} ===')
    print(f'{"Metrik":<12} {"Precision":>10} {"Recall":>10} {"F1":>10}')
    print('-' * 44)
    for metric, vals in scores.items():
        print(f'{metric:<12} {vals["P"]:>10.4f} {vals["R"]:>10.4f} {vals["F1"]:>10.4f}')


print('ROUGE evaluator ready.')


In [ ]:
# Cari file referensi
raw_dir = BASE_DIR / 'data' / 'dataset'
ref_files = sorted(raw_dir.glob('*.md')) if raw_dir.exists() else []
print(f'Referensi PRD: {len(ref_files)} file\n')

for rf in ref_files[:2]:  # evaluasi max 2 file (setiap file ~70-80 detik)
    ref_text = rf.read_text()
    ref_title = rf.stem.replace('_', ' ').title()
    print(f'{"="*60}')
    print(f'Evaluasi: {ref_title}')
    print(f'{"="*60}')

    prompt_gen = f"Buat PRD seperti contoh {ref_title}"
    print(f'Generate PRD...', end=' ', flush=True)
    t0 = time.time()
    hasil = generate_prd(prompt_gen, template_key='master')
    elapsed = time.time() - t0
    print(f'done ({elapsed:.1f}s, {len(hasil)} chars)')

    scores = evaluate_rouge(hasil, ref_text)
    print_rouge_table(scores)
    print(f'\n  Reference: {len(ref_text):,} chars')
    print(f'  Output AI: {len(hasil):,} chars')

print(f'\n{"="*60}')
print('Evaluasi ROUGE selesai.')
print(f'{"="*60}')


---
## 7. Ringkasan Alur AI

```
[Data Mentah]  -> [Chunking]  -> [Embedding] -> [ChromaDB]
 data/dataset/*.md    800 chars     MiniLM-L6-v2    Vector store
                                                      |
[Input User] -> [Intent Detect] -> [RAG Retrieve] -> [Prompt + Template] -> [LLM] -> [PRD Output]
 "Buat PRD..."   generate/qna      Cari relevan    Referensi+Template    Llama    PRD jadi
                                                      3.2 1B
```

**Komponen:**
- Data Ingestion: download, chunking, embedding, ChromaDB
- Retrieval: semantic search (RAG)
- Intent Detection: bedakan pertanyaan vs generate
- Template: 5 pilihan (master, startup, mobile, enterprise, data)
- LLM: Llama 3.2 1B Instruct
- Evaluasi: ROUGE-1/2/L

Pipeline ini identik dengan yang berjalan di backend FastAPI (`App/backend/main.py`) dan frontend React (`App/frontend/`).

In [ ]:
# Cleanup
if config.TEMP_DRIVE_DIR.exists():
    shutil.rmtree(config.TEMP_DRIVE_DIR)
    print('Temporary Drive files dihapus.')
print('Pipeline selesai.')
